# T005 — Phân tích khám phá dữ liệu (EDA) — Đề 3 (Nhóm 18)

**Tác giả:** Đỗ Minh Huấn (`huan-eda-gd`) · **Mã chấm:** B1.1, B1.3

**Nguyên tắc không rò rỉ dữ liệu:** Notebook này chỉ gọi các hàm nạp và chia tuần chuẩn trong `src/data_prep.py` và chỉ sử dụng tập huấn luyện (`train`, 209 tuần) cùng tập kiểm định (`val`, 26 tuần). Tuyệt đối **không đọc** tập kiểm tra (`test_a`, `test_b`). Mọi thống kê mô tả, ngưỡng phân vị 90% và ma trận tương quan chỉ tính trên `train`.

**Đầu ra sinh ra:**
- Bảng 1.2: `results/tables/1_2_eda_summary.csv`
- Bảng 1.3: `results/tables/1_3_eda_breakdown.csv`
- Hình 1.1: `results/figures/1_1_load_timeseries_anomalies.png`
- Hình 1.2: `results/figures/1_2_load_distribution_peak.png`
- Hình 1.3: `results/figures/1_3_diurnal_weekly_profile.png`
- Hình 1.4: `results/figures/1_4_load_vs_temperature.png`
- Hình 1.5: `results/figures/1_5_correlation_heatmap.png`

In [ ]:
# Ô 1: Thiết lập đường dẫn, nạp dữ liệu qua src/data_prep.py (không đọc tập kiểm tra)
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from matplotlib.figure import Figure

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, data_prep

df_raw = data_prep.load_raw()
splits = data_prep.split_weeks(df_raw)
train_df = splits["train"].copy()
val_df = splits["val"].copy()
train_val_df = pd.concat(
    [train_df.assign(split="train"), val_df.assign(split="val")],
    ignore_index=True,
)

y_train = train_df[config.TARGET_COL].to_numpy(dtype=np.float64)
y_val = val_df[config.TARGET_COL].to_numpy(dtype=np.float64)
peak_thr = data_prep.peak_threshold(y_train)
med_train = float(np.median(y_train))
low_thr = config.LOW_LOAD_FRACTION * med_train

config.FIG_DIR.mkdir(parents=True, exist_ok=True)
config.TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Train: {len(train_df):,} giờ ({train_df['week_idx'].nunique()} tuần), thiếu = {int(train_df.isna().sum().sum())}")
print(f"Val  : {len(val_df):,} giờ ({val_df['week_idx'].nunique()} tuần), thiếu = {int(val_df.isna().sum().sum())}")
print(f"Trung vị train = {med_train:.2f} MWh | Ngưỡng cao điểm (90%) = {peak_thr:.2f} MWh | Ngưỡng tải thấp (50% trung vị) = {low_thr:.2f} MWh")

In [ ]:
# Ô 2: Bảng 1.2 — Thống kê mô tả trên tập huấn luyện và kiểm tra giờ tải thấp bất thường
cols_summary = [config.TARGET_COL, *config.WEATHER_COLS, *config.CALENDAR_COLS]
corr_with_target = train_df[cols_summary].corr()[config.TARGET_COL]

rows = []
for col in cols_summary:
    s = train_df[col]
    rows.append({
        "bien": col,
        "n_train": int(len(s)),
        "so_thieu": int(s.isna().sum()),
        "trung_binh": round(float(s.mean()), 4),
        "do_lech_chuan": round(float(s.std(ddof=1)), 4),
        "min": round(float(s.min()), 4),
        "q25": round(float(s.quantile(0.25)), 4),
        "trung_vi": round(float(s.median()), 4),
        "q75": round(float(s.quantile(0.75)), 4),
        "max": round(float(s.max()), 4),
        "tuong_quan_nat_demand": round(float(corr_with_target[col]), 4),
    })

summary_df = pd.DataFrame(rows)
tbl_path = config.TABLE_DIR / "1_2_eda_summary.csv"
summary_df.to_csv(tbl_path, index=False)
print(f"Đã lưu Bảng 1.2: {tbl_path}")

low_df = data_prep.low_load_hours(train_val_df, ref_median=med_train).merge(
    train_val_df[[config.TIME_COL, "split"]], on=config.TIME_COL, how="left"
)
print(f"Số giờ phụ tải thấp bất thường trong train + val: {len(low_df)}")
print(low_df.to_string(index=False))

In [ ]:
# Ô 3: Hình 1.1 — Chuỗi phụ tải trung bình/cực tiểu theo tuần và 8 giờ tải thấp bất thường
weekly = (
    train_val_df.assign(
        week_start=train_val_df[config.TIME_COL]
        - pd.to_timedelta(train_val_df["hour_in_week"], unit="h")
    )
    .groupby(["split", "week_start"], as_index=False)[config.TARGET_COL]
    .agg(mean="mean", min="min", max="max")
    .sort_values("week_start")
)

fig = Figure(figsize=(12, 4.8), layout="constrained")
ax = fig.subplots()

for split_name, color, label_prefix in (
    ("train", "tab:blue", "Huấn luyện (train, 209 tuần)"),
    ("val", "tab:orange", "Kiểm định (val, 26 tuần)"),
):
    sub = weekly[weekly["split"] == split_name]
    ax.plot(sub["week_start"], sub["mean"], color=color, linewidth=1.6, label=f"TB tuần — {label_prefix}")
    ax.fill_between(sub["week_start"], sub["min"], sub["max"], color=color, alpha=0.15, linewidth=0)

# Đánh dấu 4 tuần khởi động đầu của train và ranh giới train/val
warmup_end = train_df.loc[train_df["week_idx"] == config.WARMUP_WEEKS, config.TIME_COL].min()
val_start = val_df[config.TIME_COL].min()
ax.axvspan(train_df[config.TIME_COL].min(), warmup_end, color="gray", alpha=0.18, label=f"{config.WARMUP_WEEKS} tuần khởi động (chỉ làm lịch sử)")
ax.axvline(val_start, color="black", linestyle="--", linewidth=1.1, alpha=0.8)

# Ngưỡng cao điểm 90% và ngưỡng tải thấp 50% trung vị train
ax.axhline(peak_thr, color="tab:red", linestyle=":", linewidth=1.3, label=f"Ngưỡng giờ cao điểm 90% train ({peak_thr:.2f} MWh)")
ax.axhline(low_thr, color="tab:purple", linestyle="-.", linewidth=1.3, label=f"Ngưỡng tải thấp 50% trung vị train ({low_thr:.2f} MWh)")

# Điểm tải thấp bất thường
ax.scatter(
    low_df[config.TIME_COL], low_df[config.TARGET_COL],
    color="crimson", s=45, zorder=5, marker="o",
    label=f"Giờ tải thấp bất thường (n = {len(low_df)}: 3 train, 5 val)",
)
min_row = low_df.loc[low_df[config.TARGET_COL].idxmin()]
ax.annotate(
    f"Sự cố 20/01/2019\n(thấp nhất {min_row[config.TARGET_COL]:.2f} MWh,\n{min_row['ratio_to_median']*100:.1f}% trung vị)",
    xy=(min_row[config.TIME_COL], min_row[config.TARGET_COL]),
    xytext=(min_row[config.TIME_COL] - pd.Timedelta(days=430), 220),
    arrowprops={"arrowstyle": "->", "color": "crimson", "lw": 1.2},
    fontsize=8.5, color="crimson",
    bbox={"boxstyle": "round,pad=0.3", "fc": "white", "ec": "crimson", "alpha": 0.9},
)

ax.set_title("Chuỗi phụ tải điện Panama theo tuần (train & val) và các giờ tải thấp bất thường")
ax.set_xlabel("Thời gian (tuần Thứ Bảy → Thứ Sáu)")
ax.set_ylabel("Phụ tải toàn quốc nat_demand (MWh)")
ax.set_ylim(0, 1850)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="lower left", ncol=2)

fig_path = config.FIG_DIR / "1_1_load_timeseries_anomalies.png"
fig.savefig(fig_path, dpi=130)
print(f"Đã lưu Hình 1.1: {fig_path}")

In [ ]:
# Ô 4: Hình 1.2 — Phân bố nat_demand (train) và tỷ lệ nhãn giờ cao điểm (+1 / -1)
fig = Figure(figsize=(11.5, 4.5), layout="constrained")
axes = fig.subplots(1, 2, gridspec_kw={"width_ratios": [1.5, 1.0]})

# Panel trái: phân bố nat_demand trên train
axes[0].hist(y_train, bins=45, color="steelblue", edgecolor="white", alpha=0.85, density=False)
axes[0].axvline(med_train, color="navy", linestyle="--", linewidth=1.6, label=f"Trung vị train = {med_train:.2f} MWh")
axes[0].axvline(float(np.mean(y_train)), color="darkorange", linestyle="-.", linewidth=1.6, label=f"Trung bình train = {np.mean(y_train):.2f} MWh")
axes[0].axvline(peak_thr, color="crimson", linestyle="-", linewidth=1.8, label=f"Phân vị 90% (ngưỡng cao điểm) = {peak_thr:.2f} MWh")
axes[0].axvspan(peak_thr, float(np.max(y_train)), color="crimson", alpha=0.12, label="Vùng giờ cao điểm (nhãn +1)")
axes[0].set_title(f"Phân bố phụ tải nat_demand trên tập huấn luyện (n = {len(y_train):,} giờ)")
axes[0].set_xlabel("Phụ tải toàn quốc nat_demand (MWh)")
axes[0].set_ylabel("Số giờ (tần suất)")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8.5, loc="upper left")

# Panel phải: tỷ lệ giờ cao điểm trên train và val (dùng ngưỡng tính từ train)
rate_train = float(np.mean(y_train > peak_thr) * 100.0)
rate_val = float(np.mean(y_val > peak_thr) * 100.0)
labels_sp = ["Huấn luyện\n(train, 2015–2018)", "Kiểm định\n(val, nửa đầu 2019)"]
pos_rates = [rate_train, rate_val]
neg_rates = [100.0 - r for r in pos_rates]
x_pos = np.arange(len(labels_sp))

b1 = axes[1].bar(x_pos, neg_rates, width=0.5, color="lightsteelblue", label="Giờ bình thường (nhãn −1)")
b2 = axes[1].bar(x_pos, pos_rates, bottom=neg_rates, width=0.5, color="crimson", alpha=0.85, label="Giờ cao điểm (nhãn +1)")
for i, r in enumerate(pos_rates):
    axes[1].text(i, 100.0 - r / 2.0, f"{r:.2f}%", ha="center", va="center", color="white", fontweight="bold", fontsize=10)
    axes[1].text(i, (100.0 - r) / 2.0, f"{100.0 - r:.2f}%", ha="center", va="center", color="navy", fontsize=9.5)

axes[1].set_title("Tỷ lệ nhãn 'giờ cao điểm' (ngưỡng 90% train)", fontsize=10.5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels_sp)
axes[1].set_ylabel("Tỷ lệ giờ (%)")
axes[1].set_ylim(0, 122)
axes[1].grid(True, axis="y", alpha=0.3)
axes[1].legend(fontsize=8.5, loc="upper right")

fig_path = config.FIG_DIR / "1_2_load_distribution_peak.png"
fig.savefig(fig_path, dpi=130)
print(f"Đã lưu Hình 1.2: {fig_path}")

In [ ]:
# Ô 5: Hình 1.3 — Hồ sơ phụ tải theo 24 giờ trong ngày và theo 7 ngày trong tuần (train)
t_tr = train_df[config.TIME_COL]
tr_cal = train_df.assign(
    hour=t_tr.dt.hour,
    dow=t_tr.dt.day_of_week,
    day_type=np.where(
        train_df["holiday"] == 1,
        "Ngày lễ",
        np.where(t_tr.dt.day_of_week >= 5, "Cuối tuần", "Ngày làm việc"),
    ),
)

fig = Figure(figsize=(12, 4.6), layout="constrained")
axes = fig.subplots(1, 2, gridspec_kw={"width_ratios": [1.45, 1.0]})

# Panel trái: 24 giờ theo loại ngày (TB ± 1 ĐLC)
style_map = {
    "Ngày làm việc": ("tab:blue", "-"),
    "Cuối tuần": ("tab:green", "--"),
    "Ngày lễ": ("tab:red", "-."),
}
for dt_label, (color, ls) in style_map.items():
    g = tr_cal[tr_cal["day_type"] == dt_label].groupby("hour")[config.TARGET_COL].agg(["mean", "std"])
    hrs = g.index.to_numpy()
    axes[0].plot(hrs, g["mean"], color=color, linestyle=ls, linewidth=2.0, label=f"{dt_label} (TB = {tr_cal.loc[tr_cal['day_type']==dt_label, config.TARGET_COL].mean():.1f} MWh)")
    axes[0].fill_between(hrs, g["mean"] - g["std"], g["mean"] + g["std"], color=color, alpha=0.12, linewidth=0)

axes[0].axhline(peak_thr, color="black", linestyle=":", linewidth=1.2, label=f"Ngưỡng cao điểm 90% ({peak_thr:.2f} MWh)")
axes[0].set_title("Hồ sơ phụ tải theo 24 giờ trong ngày trên train (TB ± 1 ĐLC)")
axes[0].set_xlabel("Giờ trong ngày (0–23 giờ)")
axes[0].set_ylabel("Phụ tải toàn quốc nat_demand (MWh)")
axes[0].set_xticks(range(0, 24, 2))
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8.2, loc="lower right")

# Panel phải: trung bình theo 7 ngày trong tuần theo đúng thứ tự tuần dự báo (Thứ Bảy -> Thứ Sáu)
dow_order = [5, 6, 0, 1, 2, 3, 4]
dow_names = ["T7", "CN", "T2", "T3", "T4", "T5", "T6"]
dow_stats = tr_cal.groupby("dow")[config.TARGET_COL].agg(["mean", "std"]).loc[dow_order]
bar_colors = ["tab:green", "tab:green", "tab:blue", "tab:blue", "tab:blue", "tab:blue", "tab:blue"]
x_dow = np.arange(len(dow_order))
axes[1].bar(x_dow, dow_stats["mean"], yerr=dow_stats["std"], capsize=3.5, color=bar_colors, alpha=0.85, ecolor="dimgray")
for i, (v, sd) in enumerate(zip(dow_stats["mean"], dow_stats["std"])):
    axes[1].text(i, v + sd + 18, f"{v:.0f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

axes[1].set_title("Phụ tải trung bình theo thứ (tuần T7 → T6)", fontsize=10.5)
axes[1].set_xticks(x_dow)
axes[1].set_xticklabels(dow_names)
axes[1].set_xlabel("Thứ trong tuần (xanh lá: cuối tuần, xanh dương: ngày thường)", fontsize=8.8)
axes[1].set_ylabel("Phụ tải trung bình ± 1 ĐLC (MWh)")
axes[1].set_ylim(0, 1650)
axes[1].grid(True, axis="y", alpha=0.3)

fig_path = config.FIG_DIR / "1_3_diurnal_weekly_profile.png"
fig.savefig(fig_path, dpi=130)
print(f"Đã lưu Hình 1.3: {fig_path}")

In [ ]:
# Ô 6: Hình 1.4 — Quan hệ giữa phụ tải nat_demand và nhiệt độ trung bình 3 thành phố (train)
t2m_cols = [c for c in config.WEATHER_COLS if c.startswith("T2M_")]
t_mean = train_df[t2m_cols].mean(axis=1).to_numpy(dtype=np.float64)
is_workday = (train_df["holiday"].to_numpy() == 0) & (train_df[config.TIME_COL].dt.day_of_week.to_numpy() < 5)

slope_all, intercept_all = np.polyfit(t_mean, y_train, 1)
r_all = float(np.corrcoef(t_mean, y_train)[0, 1])

fig = Figure(figsize=(11.5, 4.6), layout="constrained")
axes = fig.subplots(1, 2)

# Panel trái: biểu đồ mật độ hexbin toàn bộ tập train + hồi quy tuyến tính
hb = axes[0].hexbin(t_mean, y_train, gridsize=38, cmap="YlGnBu", mincnt=1)
t_grid = np.linspace(float(t_mean.min()), float(t_mean.max()), 100)
axes[0].plot(
    t_grid, slope_all * t_grid + intercept_all,
    color="crimson", linewidth=2.0,
    label=f"Hồi quy tuyến tính: +{slope_all:.2f} MWh/°C (r = {r_all:.3f})",
)
fig.colorbar(hb, ax=axes[0], label="Số giờ trong ô")
axes[0].set_title("Mật độ phân bố phụ tải theo nhiệt độ TB 3 thành phố (train)")
axes[0].set_xlabel("Nhiệt độ trung bình 3 thành phố T2M_mean (°C)")
axes[0].set_ylabel("Phụ tải toàn quốc nat_demand (MWh)")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8.5, loc="lower right")

# Panel phải: đường trung bình theo khoảng nhiệt độ (1°C) tách ngày làm việc và cuối tuần/lễ
bins = np.arange(np.floor(t_mean.min()), np.ceil(t_mean.max()) + 1.0, 1.0)
for mask, color, label in (
    (is_workday, "tab:blue", "Ngày làm việc"),
    (~is_workday, "tab:orange", "Cuối tuần / Ngày lễ"),
):
    tmp = pd.DataFrame({"t": t_mean[mask], "y": y_train[mask]})
    tmp["bin"] = pd.cut(tmp["t"], bins=bins)
    grp = tmp.groupby("bin", observed=True).agg(t_mid=("t", "mean"), y_mean=("y", "mean"), y_std=("y", "std"), n=("y", "size"))
    grp = grp[grp["n"] >= 20]
    axes[1].plot(grp["t_mid"], grp["y_mean"], marker="o", markersize=4, color=color, linewidth=1.8, label=label)
    axes[1].fill_between(grp["t_mid"], grp["y_mean"] - grp["y_std"], grp["y_mean"] + grp["y_std"], color=color, alpha=0.14, linewidth=0)

axes[1].axhline(peak_thr, color="crimson", linestyle=":", linewidth=1.2, label=f"Ngưỡng cao điểm 90% ({peak_thr:.2f} MWh)")
axes[1].set_title("Phụ tải TB theo dải nhiệt độ (làm việc vs cuối tuần/lễ)", fontsize=10.5)
axes[1].set_xlabel("Nhiệt độ trung bình 3 thành phố T2M_mean (°C)")
axes[1].set_ylabel("Phụ tải trung bình ± 1 ĐLC (MWh)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8.5, loc="lower right")

fig_path = config.FIG_DIR / "1_4_load_vs_temperature.png"
fig.savefig(fig_path, dpi=130)
print(f"Đã lưu Hình 1.4: {fig_path}")

In [ ]:
# Ô 7: Hình 1.5 — Ma trận tương quan Pearson trên tập huấn luyện (phát hiện đa cộng tuyến)
corr_cols = [config.TARGET_COL, *config.WEATHER_COLS, "holiday", "school"]
corr_mat = train_df[corr_cols].corr().to_numpy(dtype=np.float64)

fig = Figure(figsize=(10.5, 8.8), layout="constrained")
ax = fig.subplots()

im = ax.imshow(corr_mat, cmap="coolwarm", vmin=-1.0, vmax=1.0, aspect="auto")
fig.colorbar(im, ax=ax, shrink=0.85, label="Hệ số tương quan Pearson (r)")

ticks = np.arange(len(corr_cols))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=8.5)
ax.set_yticklabels(corr_cols, fontsize=8.5)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_mat[i, j]
        txt_color = "white" if abs(val) >= 0.55 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=txt_color, fontsize=7.2)

ax.set_title("Ma trận tương quan Pearson trên tập huấn luyện (nat_demand, 12 biến thời tiết, lịch)")
ax.set_xlabel("Biến đặc trưng và biến mục tiêu")
ax.set_ylabel("Biến đặc trưng và biến mục tiêu")

fig_path = config.FIG_DIR / "1_5_correlation_heatmap.png"
fig.savefig(fig_path, dpi=130)
print(f"Đã lưu Hình 1.5: {fig_path}")

In [ ]:
# Ô 8: Bảng 1.3 — Các chỉ số EDA phân tách (theo năm, loại ngày, tỷ lệ cao điểm, tải thấp, nhiệt độ)
breakdown_rows = []

def add_row(nhom: str, hang_muc: str, chi_so: str, gia_tri: float, don_vi: str, ghi_chu: str = "") -> None:
    breakdown_rows.append({
        "nhom": nhom,
        "hang_muc": hang_muc,
        "chi_so": chi_so,
        "gia_tri": round(float(gia_tri), 4),
        "don_vi": don_vi,
        "ghi_chu": ghi_chu,
    })

# 1. Trung bình và cực đại phụ tải theo năm (2015–2018 trên train; nửa đầu 2019 trên val)
tr_years = train_df[config.TIME_COL].dt.year
for yr in (2015, 2016, 2017, 2018):
    sub_y = train_df.loc[tr_years == yr, config.TARGET_COL]
    add_row("theo_nam", f"nam_{yr}_train", "trung_binh", sub_y.mean(), "MWh", f"n={len(sub_y)} giờ")
    add_row("theo_nam", f"nam_{yr}_train", "cuc_dai", sub_y.max(), "MWh", f"n={len(sub_y)} giờ")

add_row("theo_nam", "nam_2019_val", "trung_binh", y_val.mean(), "MWh", f"n={len(y_val)} giờ (26 tuần val)")
add_row("theo_nam", "nam_2019_val", "cuc_dai", y_val.max(), "MWh", f"n={len(y_val)} giờ (26 tuần val)")

# 2. Thống kê theo loại ngày trên train (TB, ĐLC, đỉnh, đáy, biên độ, % thấp hơn ngày làm việc)
workday_mean = float(tr_cal.loc[tr_cal["day_type"] == "Ngày làm việc", config.TARGET_COL].mean())
for dt_label, dt_code in (
    ("Ngày làm việc", "ngay_lam_viec"),
    ("Cuối tuần", "cuoi_tuan"),
    ("Ngày lễ", "ngay_le"),
):
    sub_dt = tr_cal[tr_cal["day_type"] == dt_label]
    dt_mean = float(sub_dt[config.TARGET_COL].mean())
    dt_std = float(sub_dt[config.TARGET_COL].std(ddof=1))
    hr_mean = sub_dt.groupby("hour")[config.TARGET_COL].mean()
    peak_hr, peak_val = int(hr_mean.idxmax()), float(hr_mean.max())
    trough_hr, trough_val = int(hr_mean.idxmin()), float(hr_mean.min())
    pct_lower = (workday_mean - dt_mean) / workday_mean * 100.0
    add_row("loai_ngay", dt_code, "trung_binh", dt_mean, "MWh", f"n={len(sub_dt)} giờ")
    add_row("loai_ngay", dt_code, "do_lech_chuan", dt_std, "MWh", f"n={len(sub_dt)} giờ")
    add_row("loai_ngay", dt_code, "dinh_theo_gio", peak_val, "MWh", f"lúc {peak_hr:02d}:00")
    add_row("loai_ngay", dt_code, "day_theo_gio", trough_val, "MWh", f"lúc {trough_hr:02d}:00")
    add_row("loai_ngay", dt_code, "bien_do_ngay", peak_val - trough_val, "MWh", f"{trough_hr:02d}:00 -> {peak_hr:02d}:00")
    add_row("loai_ngay", dt_code, "thap_hon_ngay_lam_viec_pct", pct_lower, "%", "so với TB ngày làm việc")

# 3. Tỷ lệ nhãn giờ cao điểm (+1) trên train và val theo ngưỡng 90% của train
n_peak_tr = int(np.sum(y_train > peak_thr))
n_peak_val = int(np.sum(y_val > peak_thr))
add_row("gio_cao_diem", "train", "nguong_90pct", peak_thr, "MWh", "phân vị 90% của train")
add_row("gio_cao_diem", "train", "ty_le_cao_diem_pct", 100.0 * n_peak_tr / len(y_train), "%", f"{n_peak_tr}/{len(y_train)} giờ")
add_row("gio_cao_diem", "val", "ty_le_cao_diem_pct", 100.0 * n_peak_val / len(y_val), "%", f"{n_peak_val}/{len(y_val)} giờ")

# 4. Giờ phụ tải thấp bất thường (< 50% trung vị train)
n_low_tr = int((low_df["split"] == "train").sum())
n_low_val = int((low_df["split"] == "val").sum())
min_low_row = low_df.loc[low_df[config.TARGET_COL].idxmin()]
add_row("tai_thap_bat_thuong", "nguong", "nguong_50pct_trung_vi_train", low_thr, "MWh", f"50% x {med_train:.2f} MWh")
add_row("tai_thap_bat_thuong", "train", "so_gio", n_low_tr, "giờ", "3 giờ đơn lẻ")
add_row("tai_thap_bat_thuong", "val", "so_gio", n_low_val, "giờ", "5 giờ liên tiếp 20/01/2019 11:00–15:00")
add_row("tai_thap_bat_thuong", "val_20_01_2019", "thap_nhat", float(min_low_row[config.TARGET_COL]), "MWh", f"{min_low_row['ratio_to_median']*100:.2f}% trung vị train")

# 5. Quan hệ phụ tải - nhiệt độ (T2M_mean) và khoảng nhiệt độ vượt ngưỡng cao điểm ngày làm việc
add_row("nhiet_do", "t2m_mean_train", "tuong_quan_r", r_all, "r", "Pearson với nat_demand")
add_row("nhiet_do", "t2m_mean_train", "do_doc_tuyen_tinh", slope_all, "MWh/°C", "hồi quy tuyến tính toàn bộ train")
tmp_work = pd.DataFrame({"t": t_mean[is_workday], "y": y_train[is_workday]})
tmp_work["bin"] = pd.cut(tmp_work["t"], bins=bins)
work_bins = tmp_work.groupby("bin", observed=True).agg(n=("y", "size"), y_mean=("y", "mean"))
work_bins = work_bins[work_bins["n"] >= 20]
for b_interval in work_bins.index:
    if b_interval.left >= 29.0 and b_interval.left <= 32.0:
        add_row(
            "nhiet_do",
            f"ngay_lam_viec_{int(b_interval.left)}_{int(b_interval.right)}C",
            "trung_binh",
            float(work_bins.loc[b_interval, "y_mean"]),
            "MWh",
            f"n={int(work_bins.loc[b_interval, 'n'])} giờ; {'vượt' if work_bins.loc[b_interval, 'y_mean'] > peak_thr else 'dưới'} ngưỡng {peak_thr:.2f}",
        )

# 6. Ví dụ định lượng sai số 1% trên train cho bài toán nghiệp vụ B1.1
add_row("bai_toan_b1_1", "sai_1pct_trung_binh_train", "do_lech_mwh", 0.01 * float(np.mean(y_train)), "MWh", f"1% của TB {np.mean(y_train):.2f} MWh")
add_row("bai_toan_b1_1", "sai_1pct_dinh_max_train", "do_lech_mwh", 0.01 * float(np.max(y_train)), "MWh", f"1% của đỉnh {np.max(y_train):.2f} MWh")

breakdown_df = pd.DataFrame(breakdown_rows)
tbl3_path = config.TABLE_DIR / "1_3_eda_breakdown.csv"
breakdown_df.to_csv(tbl3_path, index=False)
print(f"Đã lưu Bảng 1.3: {tbl3_path} ({len(breakdown_df)} dòng)")

## Tóm tắt kết quả định lượng và vấn đề dữ liệu phát hiện (B1.3)

1. **Không khuyết dữ liệu** (`0` giá trị thiếu trên cả 35.112 giờ `train` và 4.368 giờ `val`).
2. **8 giờ phụ tải thấp bất thường (`< 50%` trung vị `train` = 578,36 MWh):** 3 giờ đơn lẻ trong `train` và **5 giờ liên tiếp ngày 20/01/2019 (11:00–15:00) trong `val`** tụt sâu nhất còn **85,19 MWh** (7,4% trung vị train). *Cách xử lý:* giữ nguyên dữ liệu gốc, đánh dấu qua `data_prep.low_load_hours()` và báo MAPE trên `val` cả khi có và không có 5 giờ ngày 20/01/2019 (kèm MAE/RMSE).
3. **Xu hướng tăng tải theo năm và dịch chuyển tỷ lệ nhãn cao điểm:** Phụ tải trung bình tăng từ 1.131,52 MWh (2015) lên 1.205,68 MWh (2018) và 1.248,11 MWh (`val` nửa đầu 2019); tỷ lệ giờ cao điểm theo ngưỡng 90% của `train` (1.450,75 MWh) tăng từ **10,00%** (`train`) lên **17,54%** (`val`). *Cách xử lý:* chia theo thời gian 209/26/37/14 tuần không xáo trộn, mọi chuẩn hóa/ngưỡng chỉ fit trên `train`, bỏ 4 tuần khởi động (`data_prep.drop_warmup`) khi huấn luyện.
4. **Đa cộng tuyến mạnh giữa 3 thành phố:** Nhiệt độ `T2M` giữa Tocumen, Santiago, David có tương quan $r = 0{,}92	ext{--}0{,}96$; độ ẩm `QV2M` có $r = 0{,}82	ext{--}0{,}92$. Đây là cơ sở giải thích số điều kiện Hessian ở B3 và động lực áp dụng PCA / chọn đặc trưng ở B6.